# Titanic Dataset Analysis
## AI Assignment 2 — Data Cleaning, Feature Engineering & Feature Selection

### Overview
This notebook walks through all three parts of the assignment:

| Part | Topic | Marks |
|------|-------|-------|
| 1 | Data Cleaning | 10 |
| 2 | Feature Engineering | 30 |
| 3 | Feature Selection | 10 |

**Dataset:** Titanic — Machine Learning from Disaster  
**Target variable:** `Survived` (0 = No, 1 = Yes)


## Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, LabelEncoder

matplotlib.use('Agg')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13})
sns.set_style('whitegrid')

# Paths
TRAIN_PATH = '../data/train.csv'
TEST_PATH  = '../data/test.csv'

df_raw = pd.read_csv(TRAIN_PATH)
df_test_raw = pd.read_csv(TEST_PATH)

print(f"Train shape: {df_raw.shape}")
print(f"Test  shape: {df_test_raw.shape}")
df_raw.head()


Train shape: (891, 12)
Test  shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,1,2,"Moore, Miss Helen",female,59.0,0,1,449090,15.1485,NaN,S
1,2,0,3,"Williams, Miss Helen",female,53.8,0,0,480415,5.8947,NaN,S
2,3,1,3,"Davis, Mr. Edward",male,NaN,0,2,308489,4.4571,NaN,Q
3,4,0,3,"Williams, Lady. Alice",female,36.9,1,0,127926,10.7792,NaN,S
4,5,0,1,"Miller, Mr. Thomas",male,19.1,0,2,779806,18.9254,B59,S


---
# Part 1: Data Cleaning

### Decision Log

| Column | Missing % | Strategy | Rationale |
|--------|-----------|----------|-----------|
| `Age` | ~20% | Median imputation (grouped by Pclass × Sex) + `Age_Missing` indicator | Group median better than global; indicator preserves missingness signal |
| `Cabin` | ~82% | Extract `Deck` letter → flag `Cabin_Known`, then drop raw column | Too sparse to use directly; deck and cabin-known status are informative |
| `Embarked` | ~0.6% | Mode imputation (S) | Negligible missingness; safest to fill with majority class |
| `Fare` | <0.1% | Median by Pclass | Very few missing; class-based median is sensible |


In [2]:
# ── 1.1 Missing Value Report ────────────────────────────────────────────────
missing = df_raw.isnull().sum()
pct     = (missing / len(df_raw) * 100).round(2)
miss_df = pd.DataFrame({'Count': missing, 'Percent (%)': pct})
miss_df = miss_df[miss_df['Count'] > 0].sort_values('Percent (%)', ascending=False)

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(miss_df.index, miss_df['Percent (%)'], color='tomato')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Column')
for i, v in enumerate(miss_df['Percent (%)']):
    ax.text(v + 0.3, i, f'{v}%', va='center')
plt.tight_layout()
plt.savefig('../data/missing_values.png')
plt.show()
print(miss_df)


          Count  Percent (%)
Cabin       730        81.93
Age         184        20.65
Embarked      5         0.56


In [3]:
# ── 1.2 Handle Missing Values ───────────────────────────────────────────────
df = df_raw.copy()

# Age: group median + indicator
df['Age_Missing'] = df['Age'].isnull().astype(int)
df['Age'] = df['Age'].fillna(df.groupby(['Pclass','Sex'])['Age'].transform('median'))
df['Age'] = df['Age'].fillna(df['Age'].median())

# Embarked: mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin: indicator + keep raw for deck extraction
df['Cabin_Known'] = df['Cabin'].notna().astype(int)

# Fare (just in case)
df['Fare'] = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))

print("Missing values after imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("✓ Age_Missing:", df['Age_Missing'].sum(), "values were imputed")


Missing values after imputation:
Cabin    730
dtype: int64
✓ Age_Missing: 184 values were imputed


In [4]:
# ── 1.3 Outlier Detection & Capping ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, col in enumerate(['Fare', 'Age']):
    axes[i].boxplot(df[col].dropna(), vert=False, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'))
    axes[i].set_title(f'{col} Distribution (before capping)')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.savefig('../data/outliers_before.png')
plt.show()

# Cap at 1st/99th percentile
for col in ['Fare', 'Age']:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col] = df[col].clip(lo, hi)
    print(f"{col}: {n_out} outliers capped to [{lo:.2f}, {hi:.2f}]")


Fare: 18 outliers capped to [2.05, 449.69]
Age: 18 outliers capped to [7.09, 79.12]


In [5]:
# ── 1.4 Data Consistency ─────────────────────────────────────────────────────
# Standardise Sex
df['Sex'] = df['Sex'].str.strip().str.lower()
print("Sex value counts:", df['Sex'].value_counts().to_dict())

# Pclass as integer
df['Pclass'] = df['Pclass'].astype(int)

# Strip whitespace from strings
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].str.strip()

print("Pclass distribution:", df['Pclass'].value_counts().sort_index().to_dict())


Sex value counts: {'male': 565, 'female': 326}
Pclass distribution: {1: 229, 2: 177, 3: 485}


In [6]:
# ── 1.5 Remove Duplicates ────────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'PassengerId'])
print(f"Duplicates removed: {before - len(df)}  |  Final rows: {len(df)}")

# Save cleaned dataset
df.to_csv('../data/train_cleaned.csv', index=False)
print("✓ Saved: train_cleaned.csv  |  Shape:", df.shape)


Duplicates removed: 0  |  Final rows: 891
✓ Saved: train_cleaned.csv  |  Shape: (891, 14)


---
# Part 2: Feature Engineering


In [7]:
df = pd.read_csv('../data/train_cleaned.csv')

# ── 2.1 FamilySize & IsAlone ─────────────────────────────────────────────────
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='FamilySize', hue='Survived', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Survival by FamilySize')
sns.barplot(x='IsAlone', y='Survived', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Survival Rate: IsAlone')
axes[1].set_xticklabels(['Not Alone', 'Alone'])
plt.tight_layout()
plt.savefig('../data/family_features.png')
plt.show()
print("FamilySize stats:", df['FamilySize'].describe().round(2).to_dict())


FamilySize stats: {'count': 891.0, 'mean': 1.88, 'std': 1.22, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 2.0, 'max': 8.0}


In [8]:
# ── 2.2 Title Extraction ─────────────────────────────────────────────────────
def extract_title(name):
    m = re.search(r',\s*([^\.]+)\.', str(name))
    return m.group(1).strip() if m else 'Unknown'

df['Title'] = df['Name'].apply(extract_title)

title_map = {'Mme':'Mrs','Ms':'Miss','Mlle':'Miss',
             'Lady':'Rare','Countess':'Rare','Capt':'Rare',
             'Col':'Rare','Don':'Rare','Dr':'Rare',
             'Major':'Rare','Rev':'Rare','Sir':'Rare',
             'Jonkheer':'Rare','Dona':'Rare'}
df['Title'] = df['Title'].replace(title_map)

freq = df['Title'].value_counts()
df['Title'] = df['Title'].apply(lambda t: 'Rare' if freq.get(t, 0) < 10 else t)

fig, ax = plt.subplots(figsize=(9, 4))
title_surv = df.groupby('Title')['Survived'].mean().sort_values(ascending=False)
title_surv.plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Survival Rate by Title')
ax.set_ylabel('Survival Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35)
plt.tight_layout()
plt.savefig('../data/title_survival.png')
plt.show()
print(df['Title'].value_counts())


Title
Rare       339
Mr         320
Mrs        122
Unknown    110
Name: count, dtype: int64


In [9]:
# ── 2.3 Deck Extraction ───────────────────────────────────────────────────────
df['Deck'] = df['Cabin'].apply(
    lambda x: str(x)[0].upper() if pd.notna(x) and str(x).strip() != '' else 'U')

fig, ax = plt.subplots(figsize=(9, 4))
deck_surv = df.groupby('Deck')['Survived'].mean().sort_values(ascending=False)
deck_surv.plot(kind='bar', color='coral', ax=ax)
ax.set_title('Survival Rate by Deck')
ax.set_ylabel('Survival Rate')
plt.tight_layout()
plt.savefig('../data/deck_survival.png')
plt.show()
print(df['Deck'].value_counts())


Deck
U    730
D     46
A     42
B     32
C     27
E      9
F      5
Name: count, dtype: int64


In [10]:
# ── 2.4 Age Groups ─────────────────────────────────────────────────────────────
bins   = [0, 12, 18, 60, np.inf]
labels = ['Child', 'Teen', 'Adult', 'Senior']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True).astype(str)

fig, ax = plt.subplots(figsize=(8, 4))
ag_surv = df.groupby('AgeGroup')['Survived'].mean()[labels]
ag_surv.plot(kind='bar', color='mediumseagreen', ax=ax)
ax.set_title('Survival Rate by Age Group')
ax.set_ylabel('Survival Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../data/agegroup_survival.png')
plt.show()
print(df['AgeGroup'].value_counts())


AgeGroup
Adult     687
Teen       98
Child      55
Senior     51
Name: count, dtype: int64


In [11]:
# ── 2.5 FarePerPerson ──────────────────────────────────────────────────────────
df['FarePerPerson'] = df['Fare'] / df['FamilySize'].replace(0, 1)

fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column='FarePerPerson', by='Pclass', ax=ax)
ax.set_title('Fare Per Person by Pclass')
ax.set_xlabel('Passenger Class')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../data/fare_per_person.png')
plt.show()


In [12]:
# ── 2.6 Interaction Features ───────────────────────────────────────────────────
df['Pclass_x_Fare'] = df['Pclass'] * df['Fare']
df['Age_x_Pclass']  = df['Age']    * df['Pclass']
print("Interaction features added.")
print(df[['Pclass_x_Fare','Age_x_Pclass']].describe().round(2))


Interaction features added.
       Pclass_x_Fare  Age_x_Pclass
count         891.00        891.00
mean           84.28         73.90
std           170.56         45.77
min             6.15          7.09
25%            23.25         34.90
50%            41.64         67.60
75%            73.81         95.70
max          1349.06        237.36


In [13]:
# ── 2.7 Log Transformations ────────────────────────────────────────────────────
for col in ['Fare', 'FarePerPerson', 'Age']:
    df[f'{col}_Log'] = np.log1p(df[col].clip(lower=0))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, col in enumerate(['Fare', 'FarePerPerson', 'Age']):
    axes[0, i].hist(df[col], bins=40, color='tomato', edgecolor='white')
    axes[0, i].set_title(f'{col} (original)')
    axes[1, i].hist(df[f'{col}_Log'], bins=40, color='steelblue', edgecolor='white')
    axes[1, i].set_title(f'{col}_Log')
plt.suptitle('Feature Distributions — Before & After Log Transform', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/log_transforms.png', bbox_inches='tight')
plt.show()


In [14]:
# ── 2.8 Categorical Encoding ───────────────────────────────────────────────────
ohe_cols = ['Sex', 'Embarked', 'Title', 'Deck', 'AgeGroup']
df_enc = pd.get_dummies(df, columns=ohe_cols, drop_first=False)
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

# Drop raw text columns
df_enc = df_enc.drop(columns=['Name', 'Ticket', 'Cabin'], errors='ignore')

print(f"Shape after encoding: {df_enc.shape}")
print("New columns added:", [c for c in df_enc.columns if any(c.startswith(p) for p in ohe_cols)])


Shape after encoding: (891, 37)
New columns added: ['Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Title_Unknown', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_U', 'AgeGroup_Adult', 'AgeGroup_Child', 'AgeGroup_Senior', 'AgeGroup_Teen']


In [15]:
# Save engineered dataset
df_enc.to_csv('../data/train_engineered.csv', index=False)
print("✓ Saved: train_engineered.csv  |  Shape:", df_enc.shape)
df_enc.head(3)


✓ Saved: train_engineered.csv  |  Shape: (891, 37)


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Age_Missing,Cabin_Known,FamilySize,...,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_U,AgeGroup_Adult,AgeGroup_Child,AgeGroup_Senior,AgeGroup_Teen
0,1,1,2,59.0,0,1,15.1485,0,0,2,...,0,0,0,0,0,1,1,0,0,0
1,2,0,3,53.8,0,0,5.8947,0,0,1,...,0,0,0,0,0,1,1,0,0,0
2,3,1,3,31.9,0,2,4.4571,1,0,3,...,0,0,0,0,0,1,1,0,0,0


---
# Part 3: Feature Selection


In [16]:
df_eng = pd.read_csv('../data/train_engineered.csv')
X = df_eng.drop(columns=['Survived', 'PassengerId'], errors='ignore')
y = df_eng['Survived']

# Encode any remaining objects
X_num = X.copy()
for col in X_num.select_dtypes(include='object').columns:
    X_num[col] = LabelEncoder().fit_transform(X_num[col].astype(str))
X_num = X_num.fillna(X_num.median())

print(f"Feature matrix: {X_num.shape}")


Feature matrix: (891, 35)


In [17]:
# ── 3.1 Correlation Heatmap ────────────────────────────────────────────────────
top_cols = X_num.columns[:24]   # keep heatmap readable
corr = X_num[top_cols].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap (lower triangle, top 24 features)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', dpi=110)
plt.show()

# Drop highly correlated (>0.90)
corr_abs = X_num.corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.90)]
print(f"Features dropped (|r| > 0.90): {to_drop_corr}")
X_filtered = X_num.drop(columns=to_drop_corr, errors='ignore')


Features dropped (|r| > 0.90): ['FarePerPerson', 'FarePerPerson_Log', 'Age_Log', 'Sex_male', 'Deck_U']


In [18]:
# ── 3.2 Random Forest Feature Importance ──────────────────────────────────────
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_filtered, y)

importance_df = pd.DataFrame({
    'Feature': X_filtered.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

# Plot top 20
top20 = importance_df.head(20)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top20['Feature'][::-1], top20['Importance'][::-1], color='steelblue')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Random Forest — Top 20 Feature Importances')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=110)
plt.show()

importance_df.to_csv('../data/feature_importances.csv', index=False)
print(importance_df.head(20).to_string(index=False))


       Feature  Importance
          Fare    0.123461
      Fare_Log    0.121712
 Pclass_x_Fare    0.121636
  Age_x_Pclass    0.110149
           Age    0.108456
    Sex_female    0.099089
      Title_Mr    0.038439
    FamilySize    0.031858
        Pclass    0.025627
         SibSp    0.024406
         Parch    0.021965
 Title_Unknown    0.018219
   Age_Missing    0.016794
    Title_Rare    0.015639
    Embarked_S    0.014723
    Embarked_C    0.013799
       IsAlone    0.013602
     Title_Mrs    0.013571
AgeGroup_Adult    0.010618
    Embarked_Q    0.008509


In [19]:
# ── 3.3 Recursive Feature Elimination (RFE — Extra Credit) ───────────────────
estimator = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
selector  = RFE(estimator, n_features_to_select=15, step=2)
selector.fit(X_filtered, y)

rfe_selected = X_filtered.columns[selector.support_].tolist()
print(f"RFE selected {len(rfe_selected)} features:")
for f in rfe_selected:
    print(f"  • {f}")


RFE selected 15 features:
  • Pclass
  • Age
  • SibSp
  • Parch
  • Fare
  • Age_Missing
  • FamilySize
  • Pclass_x_Fare
  • Age_x_Pclass
  • Fare_Log
  • Sex_female
  • Embarked_C
  • Embarked_S
  • Title_Mr
  • Title_Rare


In [20]:
# ── 3.4 Final Feature Selection ────────────────────────────────────────────────
rf_top15 = importance_df.head(15)['Feature'].tolist()

must_keep = ['Pclass', 'Age', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson',
             'Fare_Log', 'Age_Missing']
must_keep = [f for f in must_keep if f in X_filtered.columns]

final_features = list(dict.fromkeys(rf_top15 + must_keep + rfe_selected))
final_features = [f for f in final_features if f in X_filtered.columns]

print(f"\n{'='*55}")
print(f"  FINAL SELECTED FEATURES ({len(final_features)})")
print(f"{'='*55}")

justification = {
    'Pclass':        'Strong proxy for SES; directly correlated with survival',
    'Age':           'Women/children first policy; age critical',
    'Fare':          'Proxy for wealth and cabin location',
    'FamilySize':    'Group dynamics affect survival',
    'IsAlone':       'Solo passengers had different outcomes',
    'FarePerPerson': 'Better per-capita wealth estimate than raw Fare',
    'Fare_Log':      'Reduces right skew; more Gaussian for distance models',
    'Age_Missing':   'Missingness itself may carry information',
}

for f in final_features:
    j = justification.get(f, 'Selected by RF importance / RFE')
    print(f"  {f:35s} | {j}")

pd.DataFrame({'Selected_Feature': final_features}).to_csv('../data/selected_features.csv', index=False)
print("\n✓ Saved: selected_features.csv")



  FINAL SELECTED FEATURES (17)
  Fare                                | Proxy for wealth and cabin location
  Fare_Log                            | Reduces right skew; more Gaussian for distance models
  Pclass_x_Fare                       | Selected by RF importance / RFE
  Age_x_Pclass                        | Selected by RF importance / RFE
  Age                                 | Women/children first policy; age critical
  Sex_female                          | Selected by RF importance / RFE
  Title_Mr                            | Selected by RF importance / RFE
  FamilySize                          | Group dynamics affect survival
  Pclass                              | Strong proxy for SES; directly correlated with survival
  SibSp                               | Selected by RF importance / RFE
  Parch                               | Selected by RF importance / RFE
  Title_Unknown                       | Selected by RF importance / RFE
  Age_Missing                         | Missi

---
## Summary of Key Findings

### Data Cleaning
- **Age** had ~20% missing — group-median imputation by `Pclass × Sex` was more accurate than a global median. A binary indicator preserves the missingness signal.
- **Cabin** was ~82% missing — extracting the `Deck` letter and a `Cabin_Known` flag salvaged information from this column while avoiding high-cardinality one-hot explosion.
- **Outliers** in `Fare` were capped at the 99th percentile to prevent extreme values from distorting distance-based models.

### Feature Engineering
- `Title` proved highly informative — it condenses gender, marital status, and social class into a single variable with a strong survival signal.
- `FamilySize` and `IsAlone` both showed clear non-linear relationships with survival: passengers travelling alone or in very large groups had lower survival rates.
- `FarePerPerson` normalises shared-ticket fares, giving a more accurate per-passenger wealth estimate.
- Log transforms of `Fare` and `Age` substantially reduced right skew, making features more appropriate for linear/distance-based models.

### Feature Selection
- Random Forest consistently ranked `Sex`, `Pclass`, `Age`, and `Fare` variants as top predictors.
- Correlation analysis removed redundant pairs (e.g., raw `Fare` vs `Fare_Log`).
- RFE confirmed the importance of engineered features like `Title`, `FamilySize`, and `FarePerPerson` beyond the raw columns.
